In [ ]:
# Purpose - tries to make field line tracking work

findex = 25 #6 #25

In [ ]:
# Imports and Folder Selection

import sys

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from graphet import Data
from graphet.plugins import TristanV2 

sys.path.append('..')
from useful_funcs import *
sys.path.append('../fmgmt')
from TristanRun import TristanRun
#from FieldLineHelper import FieldLineHelper

plt.close('all')
%matplotlib widget
set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
boilerplate = run.boilerplate #run.boilerplate + f"tstep = {run.steps[0]}"

my_steps = run.steps[:-1]

In [ ]:
# Configure Data

d = run.config_data(my_steps, "fields")

In [ ]:
def decimal_index(A, coord):
    xlen, ylen = A.shape
    return A[int(coord[0] * xlen), int(coord[1] * ylen)]

class FieldLineHelper:
    def __init__(self, run, N, contour_ds = 1, origin = [0.0, 0.0]):
        self.RUN = run
        self.N = N
        self.levels = np.full(N, np.inf)
        self.gauge_integral = 0
        self.ADJ = 0.05
        self.contour_ds = contour_ds
        self.lspace = 0

        self.origin = origin

        if not (0 <= self.origin[0] <= 1) or not (0 <= self.origin[1] <= 1):
            raise ValueError("Origin coordinates must be between 0 and 1")

    def calculate_levels(self, az_min, az_max):
        #print(self.levels)
        #print(az_min.values)
        #print(az_max.values)
        self.levels = [level for level in self.levels if level <= az_max]

        if (self.lspace == 0 or len(self.levels) == 0):
            if (self.lspace != 0 and len(self.levels) == 0):
                print("All levels gone")

            left = az_min + self.ADJ * (az_max - az_min)
            right = az_max - self.ADJ * (az_max - az_min)

            self.levels = np.linspace(left, right, self.N)
            self.lspace = self.levels[1] - self.levels[0]
        

        while (self.levels[0] - self.lspace > az_min):
            self.levels = np.insert(self.levels, 0, self.levels[0] - self.lspace)

    def plot_field_lines(self, roi, Ez_representative, arrow_spacing = 150):

        ds_roi = roi.coarsen(x=self.contour_ds, y=self.contour_ds, boundary='trim').mean()

        Azp = self.compute_Az(ds_roi) + self.gauge_integral

        self.calculate_levels(np.min(Azp), np.max(Azp))

        contours = plt.contour(Azp.x, Azp.y, Azp, levels = np.sort(self.levels), colors = 'white', linestyles = 'solid')

        if (arrow_spacing != -1):
            for collection in contours.collections:
                for path in collection.get_paths():
                    verts = path.vertices
                    if len(verts) > 1:  # Only add arrows for lines with multiple points
                        # Add an arrow marker at every nth point to avoid overcrowding
                        for i in range(0, len(verts) - 1, arrow_spacing):
                            dx = verts[i+1, 0] - verts[i, 0]
                            dy = verts[i+1, 1] - verts[i, 1]
                            plt.annotate('', xy=verts[i+1], xytext=verts[i],
                                        arrowprops=dict(arrowstyle='->', color='white', lw=0.8))   
 
        self.gauge_integral -= self.RUN.cc * self.RUN.output_interval * Ez_representative

    def compute_Az(self, ds_roi):    
        Az1 = np.cumsum(ds_roi.bx, axis=0) - np.cumsum(ds_roi.by, axis=1)
        Az2 = decimal_index(Az1, self.origin)

        return (Az1 - Az2) * self.contour_ds * self.RUN.istep

In [ ]:
dist = run.cc * run.beta_A * run.output_interval / run.c_omp
print(dist)
print(dist * 0.1)

In [ ]:
# Plot Domain and Field Lines

counter = 0
flh = FieldLineHelper(run, 10, contour_ds = 10, origin = [0, 0])

#cant skip, helper really ought to read the timestep
plt.close('all')

skip = 2
for abs_ts in range(53, 90, skip): #range(53, 150, 1): #150 #[14, 15, 16, 17]:
    plt.close('all')

    rect = [0, run.xlim, 0, run.ylim]
    #rect = [15, 65, 120, 130]

    roi = fq_cut(d.fields, abs_ts - my_steps[0], rect)
    
    plot_fq3(roi, roi.dens, vabsmax = 10, downsample = 10) #50
    #plot_fq3(roi, Azp) #Az' = Az + gauge_integral (note sign flip relative to what Robbie sent)


    ez_representative = roi.ez[0, 0]
    flh.plot_field_lines(roi, skip * ez_representative, arrow_spacing = -1)

    plt.suptitle(run.boilerplate + f"\n $t \Omega$ =  {run.gyrofreq * abs_ts:.2f} ({abs_ts})")

    plt.xlim(np.array(rect[:2]) + 3 * np.array([1, -1]))
    plt.ylim(rect[2:])
    plt.show()

    save_movie_frame(counter)
    counter += 1